<a href="https://colab.research.google.com/github/dgylayse/AkademiQ_DataScience/blob/main/AkademiQ_Data_Science_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mlflow scikit-learn pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132

In [ ]:
import mlflow
import sklearn
print("MLFlow", mlflow.__version__)
print("sklearn", sklearn.__version__)

MLFlow 3.14.0
sklearn 1.6.1


In [ ]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
mlflow.set_tracking_uri("mlruns") # Using a local directory for tracking
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: mlruns


#Experiment Oluşturma

In [ ]:
mlflow.set_experiment("musteri_rfm_model_comparison")
print("Experiment hazır")


2026/07/17 14:57:41 INFO mlflow.tracking.fluent: Experiment with name 'musteri_rfm_model_comparison' does not exist. Creating a new experiment.


Experiment hazır


In [ ]:
import mlflow.sklearn
import subprocess, time, requests

subprocess.Popen(
    ["mlflow", "ui", "--host", "127.0.0.1", "--port", "5000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)


try:
  r = requests.get("http://127.0.0.1:5000")
  print("Sunucu çalışıyor, status", r.status_code)
except:
  print("Sunucu henüz hazır değil.")

Sunucu çalışıyor, status 200


In [ ]:
!pip install pyngrok

from pyngrok import ngrok

ngrok.kill
public_url = ngrok.connect(5000)
print("MLFlow UI:", public_url)

In [ ]:
mlflow.set_experiment("musteri-kayip-riski")
print("Experiment hazır")

Experiment hazır


In [ ]:
import pickle
import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Veri oku
df = pd.read_csv("musteri_rfm_dt.csv", sep=";")

# Encoding
df["kanal_enc"] = LabelEncoder().fit_transform(df["dominant_kanal"])
df["sehir_enc"] = LabelEncoder().fit_transform(df["sehir"])

# Özellikler
features = [
    "recency_gun",
    "frequency",
    "monetary",
    "ort_sepet",
    "kanal_enc",
    "sehir_enc"
]

X = df[features]
y = df["kayip_riski"]

# Eğitim/Test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Ölçeklendirme
scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Modeli yükle
with open("svm_model.pkl", "rb") as f:
    svm = pickle.load(f)

# MLflow
with mlflow.start_run(run_name="svm-rbf-baseline"):

    y_pred = svm.predict(X_test_sc)

    if hasattr(svm, "predict_proba"):
        y_prob = svm.predict_proba(X_test_sc)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
        mlflow.log_metric("auc", auc)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("f1", f1)

    if hasattr(svm, "kernel"):
        mlflow.log_param("kernel", svm.kernel)

    if hasattr(svm, "C"):
        mlflow.log_param("C", svm.C)

    if hasattr(svm, "gamma"):
        mlflow.log_param("gamma", svm.gamma)

    mlflow.log_param("test_size", 0.2)

    mlflow.log_artifact("svm_model.pkl")
    mlflow.sklearn.log_model(svm, "svm_model")

print("MLflow kaydı başarıyla tamamlandı.")

2026/07/17 15:48:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


MLflow kaydı başarıyla tamamlandı.


In [ ]:
ip = requests.get("https://api.ipify.org").text
print("Colab IP", ip)

In [ ]:
import requests
import subprocess
import time

!npm install -g localtunnel

ip = requests.get("https://api.ipify.org").text
tunnel = subprocess.Popen(
    ["lt","--port","5000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)
output = tunnel.stdout.readline().decode()
print("MLFlow UI:",output.strip())

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 3s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋MLFlow UI: your url is: https://four-olives-lick.loca.lt
